# Random geometric network → zarr-vectors → Neuroglancer

End-to-end demo for the `zarrvectors://` fileserver. Five stages:

1. Build a 10K-node 3D geometric random graph with NetworkX.
2. Preview it interactively with Plotly 3D, coloured by hops from the cube centre.
3. Write it to a `.zarrvectors` store via `zarr_vectors.types.graphs.write_graph`.
4. Boot `LocalNeuroglancer` and load the store as two layers (nodes + edges).
5. Configure cross-section depth (cone-fade slab) and observe LOD level switching.

**Prerequisites**

- `zv-ngtools` installed editable (`pip install -e .` from this repo's root)
- `zarr-vectors` installed editable (`pip install -e .` from your local checkout)
- `networkx`, `plotly`, `nbformat>=4.2.0` available in the same env

## Stage 1 — Build a 10K-node geometric random graph

In [10]:
import networkx as nx
import numpy as np

N_NODES = 1_000
DIM = 3
RADIUS = 0.125   # tuned for ~30K edges at N=10K, dim=3
SEED = 42

G = nx.random_geometric_graph(N_NODES, radius=RADIUS, dim=DIM, seed=SEED)

positions = np.array(
    [G.nodes[i]["pos"] for i in range(N_NODES)],
    dtype=np.float32,
)
positions *= 1000.0   # scale [0,1] -> [0,1000] world units for nicer chunk extents

edges = np.array(list(G.edges()), dtype=np.int64)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Bounding box: {positions.min(0)} -> {positions.max(0)}")

Nodes: 1000
Edges: 3477
Bounding box: [2.6757226 0.4059397 0.6759315] -> [999.89954 999.90784 999.5724 ]


## Stage 2 — Plotly 3D preview

Lifts the classic NetworkX `random_geometric_graph` example into 3D: pick the node nearest the cube centre, colour every other node by its shortest-path distance (in hops) from there. Edges are subsampled and rendered in grey so the colour gradient on the nodes reads cleanly.

In [ ]:
import plotly.graph_objects as go

# Pick the node nearest the centroid (works in any dimension).
center_target = positions.mean(axis=0)
ncenter = int(np.argmin(np.sum((positions - center_target) ** 2, axis=1)))

# Hops from the centre, per node. Nodes in a different component get -1.
path_len = nx.single_source_shortest_path_length(G, ncenter)
node_hops = np.full(len(positions), -1, dtype=np.int32)
for n, d in path_len.items():
    node_hops[n] = d
reachable = node_hops >= 0

# Subsample edges; nodes are cheap enough at 10K to render in full.
EDGE_PLOT_FRAC = 0.10
rng = np.random.default_rng(SEED)
edge_idx = rng.choice(len(edges), int(len(edges) * EDGE_PLOT_FRAC), replace=False)

dim = positions.shape[1]

# Build a single line trace for the edges using NaN separators.
edge_xyz = np.full((len(edge_idx) * 3, dim), np.nan, dtype=np.float32)
edge_xyz[0::3] = positions[edges[edge_idx, 0]]
edge_xyz[1::3] = positions[edges[edge_idx, 1]]


def _xyz_kwargs(arr):
    """Map (N, 2) -> dict(x=..., y=...); (N, 3) -> dict(x=..., y=..., z=...)."""
    out = dict(x=arr[:, 0], y=arr[:, 1])
    if dim >= 3:
        out["z"] = arr[:, 2]
    return out


Scatter = go.Scatter3d if dim >= 3 else go.Scatter

fig = go.Figure(data=[
    Scatter(
        **_xyz_kwargs(edge_xyz),
        mode="lines",
        line=dict(color="lightgrey", width=1),
        opacity=0.3,
        name=f"edges (sample of {len(edges)})",
        hoverinfo="skip",
    ),
    Scatter(
        **_xyz_kwargs(positions[reachable]),
        mode="markers",
        marker=dict(
            size=2,
            color=node_hops[reachable],
            colorscale="Reds",
            reversescale=True,
            showscale=True,
            colorbar=dict(title="hops from centre"),
        ),
        name="reachable",
        hoverinfo="skip",
    ),
    Scatter(
        **_xyz_kwargs(positions[~reachable]),
        mode="markers",
        marker=dict(size=1.5, color="lightgrey", opacity=0.3),
        name="unreachable",
        hoverinfo="skip",
    ),
])

layout_kwargs = dict(
    margin=dict(l=0, r=0, t=30, b=0),
    title=(
        f"Random geometric network ({dim}D) — node colour = hops from centre "
        f"(node {ncenter})"
    ),
)
if dim >= 3:
    layout_kwargs["scene"] = dict(aspectmode="cube")
else:
    layout_kwargs["yaxis"] = dict(scaleanchor="x", scaleratio=1)

fig.update_layout(**layout_kwargs)
fig.show()

## Stage 3 — Write a `.zarrvectors` store

Graph data goes through `zarr_vectors.types.graphs.write_graph`, which produces a single store with geometry type `graph`. See `docs/tutorials/data_types/graphs_skeletons.md` in the `zarr-vectors-py` checkout for the full parameter reference.

In [20]:
import shutil
from pathlib import Path
from zarr_vectors.types.graphs import write_graph

STORE_PATH = Path.cwd() / "random_network.zarrvectors"
if STORE_PATH.exists():
    shutil.rmtree(STORE_PATH)

# Positions span [0, 1000]; 100^3 chunks -> 10x10x10 grid.
result = write_graph(
    str(STORE_PATH),
    positions=positions,
    edges=edges,
    chunk_shape=(100.0, 100.0, 100.0),
    bin_shape=(25.0, 25.0, 25.0),
    is_tree=False,
    dtype="float32",
)
print(f"Store: {STORE_PATH}")
print(result)

ERROR:ngtools.local.zarrvectors:failed to open zarr-vectors store
Traceback (most recent call last):
  File "/home/andrew/scripts/zv-ngtools/ngtools/local/zarrvectors.py", line 1925, in get
    store = _open_store(protocol, store_path)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/andrew/scripts/zv-ngtools/ngtools/local/zarrvectors.py", line 1195, in _open_store
    return _default_open_store(protocol, store_path)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/andrew/scripts/zv-ngtools/ngtools/local/zarrvectors.py", line 1188, in _default_open_store
    zvr = open_zvr(full)
          ^^^^^^^^^^^^^^
  File "/home/andrew/scripts/zarr-vectors-py/zarr_vectors/lazy/store.py", line 153, in open_zvr
    root = open_store(str(path))
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/andrew/scripts/zarr-vectors-py/zarr_vectors/core/store.py", line 333, in open_store
    raise StoreError(
zarr_vectors.exceptions.StoreError: Not a valid ZVF store: missing 'zarr_vecto

Store: /home/andrew/scripts/zv-ngtools/examples/random_network.zarrvectors
{'node_count': 1000, 'edge_count': 3477, 'chunk_count': 643, 'intra_edge_count': 473, 'cross_edge_count': 3004, 'object_count': 1, 'is_tree': False}


## Stage 4 — View in Neuroglancer

`LocalNeuroglancer` boots a Tornado fileserver and a Neuroglancer viewer in your default browser. `viewer.load("zarrvectors://...")` registers the graph store as two annotation layers — `random_network.zarrvectors/nodes` (POINT) and `random_network.zarrvectors/edges` (LINE) — both routed through the local fileserver as `precomputed://` annotation sources.

In [13]:
from ngtools.local.viewer import LocalNeuroglancer

viewer = LocalNeuroglancer()
print(f"fileserver:   {viewer.get_fileserver_url()}")
print(f"neuroglancer: {viewer.get_viewer_url()}")

fileserver:   http://127.0.0.1:54733/
neuroglancer: http://127.0.0.1:54235/v/1/


In [14]:
import webbrowser

zv_url = f"zarrvectors://{STORE_PATH}"
print(f"Loading: {zv_url}")
viewer.load(zv_url)

webbrowser.open(viewer.get_viewer_url())

Loading: zarrvectors:///home/andrew/scripts/zv-ngtools/examples/random_network.zarrvectors
Loaded (zarrvectors): random_network.zarrvectors/nodes
Loaded (zarrvectors): random_network.zarrvectors/edges


True

## Stage 5 — Standard 4-panel + behind-slice clip + cursor-centred chunk LOD

Layout is the **standard `"4panel"`** (xy / xz / yz cross-sections + one 3D panel) so the cross-section panels stay linked: scrolling the slice in one updates the others.

Three modifications layered on top:

1. **Wide cross-section depth.** `state.cross_section_depth = 2000` (≥ volume diagonal). Neuroglancer's hard-coded `getCrossSectionFadeFactor` still applies, but its falloff is now stretched over the whole volume, so the practical fade across any visible region is gentle.
2. **Behind-slice clip via shader.** Each LOD layer's shader transforms the annotation's `(x, y, z)` through `uModelViewProjection` to get clip-space depth (the slice plane is at `clip.z = 0`). When `PROJECTION_VIEW == false` (cross-section panels) and the annotation is on the camera side (`clip.z < 0`), it's discarded. The 3D panel renders everything. Since `uModelViewProjection` includes the current slice position, the same shader works for every cross-section orientation without per-panel configuration.
3. **Best-effort transparent slice background.** `state.cross_section_background_color = "#00000000"`. Neuroglancer's behaviour with alpha=0 is implementation-dependent — on some builds it renders as transparent, on others as opaque black. There is no public state knob beyond this.

Cursor-centred chunk LOD (8 layers, decimated 1, 1/2, 1/4, 1/8) is unchanged from the previous iteration. It still works in this layout — each cross-section panel shows the cursor-centred rings filtered to that panel's behind-slice half.

In [15]:
# Stage 5a — standard 4-panel layout, wide depth, transparent slice background.
#
# 4panel = xy + xz + yz cross-section panels + one 3D panel, all linked
# (moving the slice in one updates the others).
#
# cross_section_depth = 2000 spans the full volume diagonal so Neuroglancer's
# native distance-from-slice fade is stretched across the whole visible range
# (= gentle, near-invisible falloff in practice).
#
# cross_section_background_color = "#00000000" requests a transparent slice
# background. On some Neuroglancer builds this renders as opaque black; the
# behind-slice annotations drawn on top by the shader are still visible
# either way.

with viewer.txn() as state:
    state.layout = "4panel"
    state.position = [500.0, 500.0, 500.0]
    state.cross_section_depth = 2000.0
    state.cross_section_background_color = "#00000000"

    print(f"layout = {state.layout}")
    print(f"position = {list(state.position)}")
    print(f"cross_section_depth = {state.cross_section_depth}")
    print(f"cross_section_background_color = {state.cross_section_background_color}")

layout = DataPanelLayout("4panel")
position = [np.float32(500.0), np.float32(500.0), np.float32(500.0)]
cross_section_depth = 2000.0
cross_section_background_color = #00000000


### Stage 5b — verify the default shader is in place

The package now applies a depth-fade annotation shader **automatically** to every `zarrvectors://` layer at load time (see `_DEFAULT_SHADERS_BY_DATATYPE` and the `default_shader` field in `_make_layer_spec`, both in `ngtools/local/zarrvectors.py`; applied in `Scene._register_zv_layer`).

The shader has three sliders, exposed in each layer's Render tab — so you can tune without re-running this cell:

- `clip_sign` (±1) — flip if your build's depth sign convention is reversed.
- `fade_in_front` (NDC) — fade ramp in front of the slice. Default `0.6` (≈ 600 voxels with `cross_section_depth = 2000`).
- `fade_behind` (NDC) — fade ramp behind the slice. Default `10.0` (effectively no fade).

The cell below just checks the shader landed correctly. Tune via the Render tab to taste.

In [ ]:
# Stage 5b — verify the default shader is in place on each layer.
# (No need to set a shader here — Scene._register_zv_layer did it at load.)

with viewer.txn() as state:
    for name in ("random_network.zarrvectors/nodes",
                 "random_network.zarrvectors/edges"):
        layer = state.layers[name].layer
        s = layer.shader or ""
        ok = ("uModelViewProjection" in s and "fade_in_front" in s
              and "fade_behind" in s)
        print(f"  {name}: shader {len(s)} chars, fade controls "
              f"{'present' if ok else 'MISSING'}")

In [17]:
# Stage 5c — sync_cursor() helper.
#
# Neuroglancer doesn't propagate state.position into shader_controls
# automatically, so call this after navigating to refresh the cursor
# centre across all 8 LOD layers.

LOD_LAYER_NAMES = [
    f"random_network.zarrvectors/{kind}_lod{lod}"
    for kind in ("nodes", "edges")
    for lod in range(4)
]


def sync_cursor():
    with viewer.txn() as state:
        x, y, z = (float(v) for v in state.position)
        for name in LOD_LAYER_NAMES:
            layer = state.layers[name].layer
            controls = dict(layer.shader_controls or {})
            controls.update({"cursor_x": x, "cursor_y": y, "cursor_z": z})
            layer.shader_controls = controls
        return (x, y, z)


print(f"sync_cursor() -> {sync_cursor()}")

sync_cursor() -> (500.0, 500.0, 500.0)
